# L6: Multi-agent Collaboration for Financial Analysis

In this lesson, you will learn ways for making agents collaborate with each other.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [2]:
from crewai import Agent, Task, Crew

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [3]:
from utils import load_env
load_env()

## crewAI Tools

In [4]:
from utils import BaiduWebSearchTool, ScrapeWebsiteTool

search_tool = BaiduWebSearchTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

In [5]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="实时监控并分析市场数据，"
         "识别趋势并预测市场走势。",
    backstory="该 Agent 专精金融市场，"
              "运用统计建模与机器学习提供关键洞察。"
              "凭借对数据的敏锐把握，"
              "数据分析师 Agent 是支撑交易决策的基石。",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [6]:
trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="基于数据分析师 Agent 的洞察"
         "开发并测试多种交易策略。",
    backstory="该 Agent 深谙金融市场与量化分析，"
              "负责制定并优化交易策略。"
              "它评估不同方案的表现，"
              "以确定收益最高、风险最可控的选项。",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [7]:
execution_agent = Agent(
    role="Trade Advisor",
    goal="根据已批准的交易策略"
         "提出最优交易执行方案。",
    backstory="该 Agent 专精于分析潜在交易的时机、"
              "价格与后勤细节。通过评估这些因素，"
              "它为何时、如何执行交易提供有据可依的建议，"
              "以最大化效率并符合策略要求。",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [8]:
risk_management_agent = Agent(
    role="Risk Advisor",
    goal="评估潜在交易活动的风险"
         "并提供相关洞察。",
    backstory="该 Agent 深谙风险评估模型与市场动态，"
              "仔细审视拟议交易的潜在风险。"
              "它提供风险敞口的详细分析，"
              "并建议保障措施，确保交易活动"
              "符合公司的风险承受能力。",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

## Creating Tasks

In [9]:
# Task for Data Analyst Agent: Analyze Market Data
data_analysis_task = Task(
    description=(
        "持续监控并分析所选股票（{stock_selection}）的市场数据。"
        "运用统计建模与机器学习识别趋势并预测市场走势。"
    ),
    expected_output=(
        "关于 {stock_selection} 重大市场机会"
        "或威胁的洞察与预警。"
    ),
    agent=data_analyst_agent,
)

In [10]:
# Task for Trading Strategy Agent: Develop Trading Strategies
strategy_development_task = Task(
    description=(
        "根据数据分析师的洞察及用户设定的风险承受能力（{risk_tolerance}），"
        "制定并优化交易策略。"
        "考虑交易偏好（{trading_strategy_preference}）。"
    ),
    expected_output=(
        "一套针对 {stock_selection} 的潜在交易策略，"
        "与用户的风险承受能力相匹配。"
    ),
    agent=trading_strategy_agent,
)


In [11]:
# Task for Trade Advisor Agent: Plan Trade Execution
execution_planning_task = Task(
    description=(
        "分析已批准的交易策略，结合当前市场状况与最优定价，"
        "为 {stock_selection} 确定最佳执行方式。"
    ),
    expected_output=(
        "详细的执行计划，说明如何以及何时"
        "对 {stock_selection} 执行交易。"
    ),
    agent=execution_agent,
)


In [12]:
# Task for Risk Advisor Agent: Assess Trading Risks
risk_assessment_task = Task(
    description=(
        "评估 {stock_selection} 拟议交易策略与执行计划的相关风险。"
        "提供潜在风险的详细分析并提出缓解策略。"
    ),
    expected_output=(
        "为{stock_selection}制定全面的风险分析报告"
        "根据潜在风险及缓解建议。"
    ),
    agent=risk_management_agent,
)

## Creating the Crew
- The `Process` class helps to delegate the workflow to the Agents (kind of like a Manager at work)
- In the example below, it will run this hierarchically.
- `manager_llm` lets you choose the "manager" LLM you want to use.

In [13]:
from crewai import Crew, Process
from langchain_openai import ChatOpenAI

# Define the crew with agents and tasks
financial_trading_crew = Crew(
    agents=[data_analyst_agent, 
            trading_strategy_agent, 
            execution_agent, 
            risk_management_agent],
    
    tasks=[data_analysis_task, 
           strategy_development_task, 
           execution_planning_task, 
           risk_assessment_task],
    
    # manager_llm=ChatOpenAI(model="gpt-3.5-turbo", 
                        #    temperature=0.7),
    # process=Process.hierarchical,
    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [14]:
# Example data for kicking off the process
financial_trading_inputs = {
    'stock_selection': 'AAPL',
    'initial_capital': '100000',
    'risk_tolerance': 'Medium',
    'trading_strategy_preference': 'Day Trading',
    'news_impact_consideration': True
}

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [15]:
### this execution will take some time to run
result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.4                                                                                        │
│  Latest version:  1.14.5                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a60ce9bd-8fdf-4d92-b123-55076b5426fb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 持续监控并分析所选股票（AAPL）的市场数据。运用统计建模与机器学习识别趋势并预测市场走势。                 │
│  ID: 997ad1b7-fe29-4a4d-8f36-72c304bf67ea                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Analyst                                                                                            │
│                                                                                                                 │
│  Task: 持续监控并分析所选股票（AAPL）的市场数据。运用统计建模与机器学习识别趋势并预测市场走势。                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet                                                                                      │
│  Args: {'search_query': 'AAPL market data analysis and prediction'}                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Risk Advisor                                                                                            │
│                                                                                                                 │
│  Task: 评估 AAPL 拟议交易策略与执行计划的相关风险。提供潜在风险的详细分析并提出缓解策略。                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Risk Advisor                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # AAPL 拟议交易策略与执行计划的风险分析报告                                                                    │
│                                                                                                                 │
│  ## 1. 潜在风险分析                                                                                             │
│                                                                                                                 │
│  ### 1.1 财报结果的不确定性                                                                                     │
│  -                                                                                                              │
│  **风险描述**：尽管市场对苹果2026财年第二季度财报的预期较高，但实际结果可能与预期不符。例如，高盛和摩根大通的   │
│  预测均高于华尔街平均预期，但如果最终财报数据低于预期，股价可能大幅下跌。                                       │
│  - **影响**：可能导致投资者在财报前买入后遭遇亏损，尤其是在没有充分设置止损点的情况下。                         │
│  - **缓解策略**：                                                                                               │
│    - 设置严格的止损点（如当前价格的5%以下），以防止因财报结果不佳而遭受重大损失。                               │
│    - 在财报发布前密切关注市场情绪和分析师预测，随时准备调整策略。                                               │
│                                                                                                                 │
│  ### 1.2 iPhone 17销售表现的波动性                                                                              │
│  - **风险描述**：虽然iPhone                                                                                     │
│  17系列在中美市场表现出色，但市场需求可能受经济环境、竞争产品或消费者偏好变化的影响，导致销售数据不及预期。     │
│  - **影响**：若销售数据未达预期，AAPL股价可能受到冲击，影响交易收益。                                           │
│  - **缓解策略**：                                                                                               │
│    - 实时监控市场动态和销售数据，避免过度依赖单一消息源。                                                       │
│    - 设定严格的止损点，确保在市场出现不利变化时及时退出。                                                       │
│                                                                                                                 │
│  ### 1.3 中国市场的政策风险                                                                                     │
│  -                                                                                                              │
│  **风险描述**：苹果在中国市场的成功部分得益于加量不加价的策略，但政策变化（如补贴政策调整）可能影响其市场份额   │
│  。                                                                                                             │
│  - **影响**：若政策发生不利变化，苹果在中国市场的增长可能受阻，进而影响股价。                                   │
│  - **缓解策略**：                                                                                               │
│    - 密切关注相关政策变化，及时调整交易策略。                                                                   │
│    - 避免在政策不确定时期过度持仓，降低风险敞口。                                                               │
│                                                                                                                 │
│  ### 1.4 供应链和关税的不确定性                                                                                 │
│  - **风险描述**：苹果在第一季度承担了14亿美元的关税成本，新一轮301条款调查可能带来额外成本，影响其利润。        │
│  - **影响**：若关税政策进一步收紧，苹果的毛利率可能受到打击，从而影响股价。                                     │
│  - **缓解策略**：                                                                                               │
│    - 在关税政策不确定期间，通过买入看跌期权或做空AAPL进行对冲。                                                 │
│    - 评估对冲交

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 评估 AAPL 拟议交易策略与执行计划的相关风险。提供潜在风险的详细分析并提出缓解策略。                       │
│  Agent: Risk Advisor                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a60ce9bd-8fdf-4d92-b123-55076b5426fb                                                                       │
│  Final Output: # AAPL 拟议交易策略与执行计划的风险分析报告                                                      │
│                                                                                                                 │
│  ## 1. 潜在风险分析                                                                                             │
│                                                                                                                 │
│  ### 1.1 财报结果的不确定性                                                                                     │
│  -                                                                                                              │
│  **风险描述**：尽管市场对苹果2026财年第二季度财报的预期较高，但实际结果可能与预期不符。例如，高盛和摩根大通的   │
│  预测均高于华尔街平均预期，但如果最终财报数据低于预期，股价可能大幅下跌。                                       │
│  - **影响**：可能导致投资者在财报前买入后遭遇亏损，尤其是在没有充分设置止损点的情况下。                         │
│  - **缓解策略**：                                                                                               │
│    - 设置严格的止损点（如当前价格的5%以下），以防止因财报结果不佳而遭受重大损失。                               │
│    - 在财报发布前密切关注市场情绪和分析师预测，随时准备调整策略。                                               │
│                                                                                                                 │
│  ### 1.2 iPhone 17销售表现的波动性                                                                              │
│  - **风险描述**：虽然iPhone                                                                                     │
│  17系列在中美市场表现出色，但市场需求可能受经济环境、竞争产品或消费者偏好变化的影响，导致销售数据不及预期。     │
│  - **影响**：若销售数据未达预期，AAPL股价可能受到冲击，影响交易收益。                                           │
│  - **缓解策略**：                                                                                               │
│    - 实时监控市场动态和销售数据，避免过度依赖单一消息源。                                                       │
│    - 设定严格的止损点，确保在市场出现不利变化时及时退出。                                                       │
│                                                                                                                 │
│  ### 1.3 中国市场的政策风险                                                                                     │
│  -                                                                                                              │
│  **风险描述**：苹果在中国市场的成功部分得益于加量不加价的策略，但政策变化（如补贴政策调整）可能影响其市场份额   │
│  。                                                                                                             │
│  - **影响**：若政策发生不利变化，苹果在中国市场的增长可能受阻，进而影响股价。                                   │
│  - **缓解策略**：                                                                                               │
│    - 密切关注相关政策变化，及时调整交易策略。                                                                   │
│    - 避免在政策不确定时期过度持仓，降低风险敞口。                                                               │
│                                                                                                                 │
│  ### 1.4 供应链和关税的不确定性                                                                                 │
│  - **风险描述**：苹果在第一季度承担了14亿美元的关税成本，新一轮301条款调查可能带来额外成本，影响其利润。        │
│  - **影响**：若关税政策进一步收紧，苹果的毛利率可能受到打击，从而影响股价。                                     │
│  - **缓解策略**：                                                                                               │
│    - 在关税政策不确定期间，通过买入看跌期权或做空AAPL进行对冲。                                                 │
│    - 评估对冲

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

- Display the final result as Markdown.

In [16]:
from IPython.display import Markdown
Markdown(result.raw)

# AAPL 拟议交易策略与执行计划的风险分析报告

## 1. 潜在风险分析

### 1.1 财报结果的不确定性
- **风险描述**：尽管市场对苹果2026财年第二季度财报的预期较高，但实际结果可能与预期不符。例如，高盛和摩根大通的预测均高于华尔街平均预期，但如果最终财报数据低于预期，股价可能大幅下跌。
- **影响**：可能导致投资者在财报前买入后遭遇亏损，尤其是在没有充分设置止损点的情况下。
- **缓解策略**：
  - 设置严格的止损点（如当前价格的5%以下），以防止因财报结果不佳而遭受重大损失。
  - 在财报发布前密切关注市场情绪和分析师预测，随时准备调整策略。

### 1.2 iPhone 17销售表现的波动性
- **风险描述**：虽然iPhone 17系列在中美市场表现出色，但市场需求可能受经济环境、竞争产品或消费者偏好变化的影响，导致销售数据不及预期。
- **影响**：若销售数据未达预期，AAPL股价可能受到冲击，影响交易收益。
- **缓解策略**：
  - 实时监控市场动态和销售数据，避免过度依赖单一消息源。
  - 设定严格的止损点，确保在市场出现不利变化时及时退出。

### 1.3 中国市场的政策风险
- **风险描述**：苹果在中国市场的成功部分得益于加量不加价的策略，但政策变化（如补贴政策调整）可能影响其市场份额。
- **影响**：若政策发生不利变化，苹果在中国市场的增长可能受阻，进而影响股价。
- **缓解策略**：
  - 密切关注相关政策变化，及时调整交易策略。
  - 避免在政策不确定时期过度持仓，降低风险敞口。

### 1.4 供应链和关税的不确定性
- **风险描述**：苹果在第一季度承担了14亿美元的关税成本，新一轮301条款调查可能带来额外成本，影响其利润。
- **影响**：若关税政策进一步收紧，苹果的毛利率可能受到打击，从而影响股价。
- **缓解策略**：
  - 在关税政策不确定期间，通过买入看跌期权或做空AAPL进行对冲。
  - 评估对冲交易的成本与收益比，避免过度投机。

### 1.5 AI进展的不确定性
- **风险描述**：苹果在AI领域的进展缓慢，Siri的革新已被推迟，与谷歌Gemini的合作可能无法迅速转化为收入增长点。
- **影响**：若AI相关消息未能推动股价上涨，交易可能面临亏损。
- **缓解策略**：
  - 设定严格的止损点，避免因技术进展不及预期而遭受损失。
  - 结合基本面分析，避免过度依赖短期炒作。

### 1.6 高管变动的不确定性
- **风险描述**：新CEO约翰·特努斯的上任可能带来战略调整，市场对其未来方向存在不确定性。
- **影响**：若市场对新CEO的反应消极，AAPL股价可能受到影响。
- **缓解策略**：
  - 设置严格的止损点，以应对高管变动带来的市场波动。
  - 结合基本面分析，避免过度依赖市场情绪。

## 2. 缓解策略建议

### 2.1 严格控制仓位
- **建议**：每种交易策略均需控制仓位规模，避免过度暴露于市场波动。建议将总仓位分为3-5次，在每天的交易时段中逐步买入。
- **理由**：分散风险，减少单次交易对整体投资组合的影响。

### 2.2 设置止损点
- **建议**：为每笔交易设置明确的止损点，例如当前价格的5%以下，确保在市场出现不利波动时及时退出。
- **理由**：保护资本，避免因市场突然转向而遭受重大损失。

### 2.3 实时监控市场动态
- **建议**：使用实时市场信息工具（如财经新闻、市场报告、社交媒体等）持续监测市场动态，以便及时调整交易策略。
- **理由**：市场情绪和政策变化可能迅速影响股价，实时监控有助于把握最佳交易时机。

### 2.4 回测与模拟交易
- **建议**：在实际操作前，进行回测和模拟交易，以验证策略的有效性并优化执行细节。
- **理由**：通过历史数据测试策略，提高实际交易的成功率。

### 2.5 多策略组合
- **建议**：结合多种交易策略（如财报前趋势交易、iPhone销售业绩驱动交易、中国市场韧性交易等），形成多策略组合，降低单一策略的风险。
- **理由**：不同策略的收益和风险特征不同，组合投资可以平衡整体风险。

## 3. 总结

针对AAPL的拟议交易策略与执行计划，潜在风险主要包括财报结果的不确定性、iPhone销售表现的波动性、中国市场的政策风险、供应链和关税的不确定性、AI进展的不确定性以及高管变动的不确定性。为了有效缓解这些风险，建议采取以下措施：

- **严格控制仓位**，避免过度暴露于市场波动。
- **设置止损点**，保护资本，避免因市场突然转向而遭受损失。
- **实时监控市场动态**，及时调整交易策略。
- **进行回测与模拟交易**，验证策略的有效性。
- **采用多策略组合**，平衡整体风险。

通过以上措施，可以最大限度地降低交易风险，提高交易成功的可能性。